# 04 · Review a model on the frozen validation split
Metrics + side-by-side plots. Clear outputs before committing (nbstripout does it for you).

In [ ]:
# --- SETUP (same in every notebook) -----------------------------------
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, subprocess
REPO = "/content/pei-landuse"
GITHUB = "github.com/YOUR_USER/pei-landuse.git"     # private repo
token = userdata.get("GITHUB_TOKEN")                 # Colab ▸ Secrets (key icon). Never paste tokens in cells.
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", f"https://{token}@{GITHUB}", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{REPO}/src")
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
CONFIG = "configs/train_1968_finetune.yaml"   # any config with training.dataset
N_PLOTS = 10

import random, torch
from pathlib import Path
from pei.config import load_config
from pei.dataset import ClearLandDataset, get_split
from pei.evaluate import evaluate_model
from pei.model import load_for_inference
from pei.viz import plot_comparison

cfg = load_config(CONFIG)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds_root = Path(cfg["paths"]["training_data"]) / cfg["training"]["dataset"]
_, val_names = get_split(ds_root, ds_root / "split.json")
val_ds = ClearLandDataset(ds_root, augment=False, names=val_names)
model = load_for_inference(cfg, device)
print(evaluate_model(model, val_ds, device, cfg))

In [ ]:
with torch.no_grad():
    for i in random.sample(range(len(val_ds)), min(N_PLOTS, len(val_ds))):
        img, tgt = val_ds[i]
        plot_comparison(img, tgt, model([img.to(device)])[0], cfg, title=val_ds.names[i])